In [13]:
import os,csv 


DATA_PATH = '/home/vit/Projects/cryptoshow-analysis/data/data-extraction'
PDB_PATH = f'{DATA_PATH}/COACH420/receptor_pdb'


pockets_dict = {}
with open(f'{DATA_PATH}/COACH420/label.csv', 'r') as f:
    reader = csv.reader(f, delimiter=',')
    next(reader)  # Skip header
    for row in reader:
        protein_id = row[1]
        pockets_dict[protein_id] = []
        chain_id = protein_id[4:]
        label = row[3]
        pockets = label.split(';')
        for pocket in pockets:
            pocket = ' '.join(pocket.strip().split())
            pocket_residues = []
            for residue in pocket.strip().split(' '):
                assert residue.split('_')[0] == chain_id, f"Chain ID mismatch: {residue.split('_')[0]} != {chain_id}"
                pocket_residues.append(residue.split('_')[1])
            
            pockets_dict[protein_id].append(pocket_residues)

pockets_dict

{'148lE': [['11',
   '20',
   '22',
   '24',
   '26',
   '30',
   '32',
   '70',
   '104',
   '105',
   '106',
   '107',
   '142'],
  ['105',
   '106',
   '114',
   '115',
   '116',
   '117',
   '132',
   '135',
   '136',
   '137',
   '138',
   '141']],
 '1a26A': [['165', '229', '242', '246', '323', '324', '325', '327']],
 '1a4kH': [['31', '32', '33', '35', '50', '99', '100', '101', '104']],
 '1a7xA': [['54', '79', '81', '82', '83']],
 '1a8tA': [['82', '84', '145'],
  ['29',
   '32',
   '35',
   '36',
   '145',
   '164',
   '168',
   '175',
   '176',
   '206',
   '207',
   '208']],
 '1afkA': [['4',
   '7',
   '11',
   '12',
   '41',
   '65',
   '67',
   '69',
   '71',
   '109',
   '111',
   '118',
   '119',
   '120']],
 '1arcA': [['57',
   '169',
   '189',
   '191',
   '192',
   '194',
   '210',
   '211',
   '212',
   '214',
   '225']],
 '1atlA': [['140', '144', '150'],
  ['104',
   '105',
   '106',
   '107',
   '140',
   '141',
   '144',
   '150',
   '163',
   '165',
   '166',
   '167

### Map residues
Extract the sequence and coordinates, map the binding residues numbering to the sequence

In [ ]:
import sys
sys.path.append('/home/vit/Projects/cryptoshow-analysis/src/utils')
import cryptoshow_utils

import biotite.database.rcsb as rcsb
import biotite.structure.io.pdb as pdb
from biotite.structure.io.pdb import get_structure
from biotite.structure import get_residues
import numpy as np

translated_pockets_dict = {}
sequence_dict = {}
for protein_id, pockets in pockets_dict.items():
    translated_pockets_dict[protein_id] = []
    mapped_pocket = []
    for pocket in pockets:
        cif_file = pdb.PDBFile.read(f'{PDB_PATH}/{protein_id}.pdb')
        chain_id = protein_id[4:]
        protein = get_structure(cif_file, model=1)
        protein = protein[(protein.atom_name == "CA") 
                            & (protein.element == "C") 
                            & (protein.chain_id == chain_id) ]

        residue_ids, residue_types = get_residues(protein)
        sequence = ''
        coords = protein.coord
        np.save(f'{DATA_PATH}/coach420-coordinates/{protein_id}.npy', coords)
        for i in range(len(residue_ids)):
            residue_id = str(residue_ids[i])
            amino_acid = cryptoshow_utils.three_to_one(residue_types[i])

            if residue_id in pocket:
                mapped_pocket.append(f'{amino_acid}{i}')

            sequence += amino_acid
    translated_pockets_dict[protein_id].append(mapped_pocket)
    sequence_dict[protein_id] = sequence



In [14]:
with open(f'{DATA_PATH}/coach420_for_pocket_level_evaluation.csv', 'w', newline='') as f:
    for protein_id, pockets in translated_pockets_dict.items():
        for pocket in pockets:
            f.write(f'{protein_id};UNKNOWN;NON_CRYPTIC;{" ".join(pocket)};{sequence_dict[protein_id]}\n')
